# Notebook 3: Clustering Birdsong with Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/williamedwardhahn/birdsong/blob/main/Notebook_3_Clustering.ipynb)

In Notebook 2 you used your eyes and ears to tell bird species apart. Now we'll teach a **computer** to do the same thing.

The plan:
1. **Extract features** from each recording that capture what makes it sound unique
2. **Train an autoencoder** (a simple neural network) to compress those features into 2D
3. **Plot the results** — if it works, songs from the same species will cluster together

### Before You Start

1. Go to **File → Save a copy in Drive**
2. Rename it with your name (e.g. `FirstName_LastName_Notebook_3.ipynb`)
3. Work in **your copy** from now on

**When you're done:** Click the **Share** button (top right), copy the link, and submit it to your instructor. Make sure sharing is set to **"Anyone with the link can view."**

**Colab tip:** Close any other notebooks you have open — Colab only lets you run one or two at a time. If it disconnects, use **Runtime → Disconnect and delete runtime**, then reopen.

---
## Step 1: Setup

In [ ]:
!pip install -q librosa

import os
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder, StandardScaler
from IPython.display import Audio, display

# Set random seeds for reproducible results
torch.manual_seed(42)
np.random.seed(42)

# Download the dataset
REPO_DIR = "/content/birdsong"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/williamedwardhahn/birdsong.git {REPO_DIR}
    print("Dataset downloaded!")
else:
    print("Dataset already downloaded.")

# Find species folders — only include directories that contain .wav files
SPECIES = sorted([
    name for name in os.listdir(REPO_DIR)
    if os.path.isdir(os.path.join(REPO_DIR, name))
    and any(f.endswith('.wav') for f in os.listdir(os.path.join(REPO_DIR, name)))
])
SAMPLE_RATE = 22050
print(f"Found {len(SPECIES)} species: {SPECIES}")

## Step 2: Helper Functions

We define the autoencoder and some utilities here so the later cells stay clean.

In [ ]:
def build_autoencoder(input_dim, bottleneck_dim=2):
    """
    Build an autoencoder that compresses input_dim features
    down to bottleneck_dim dimensions, then reconstructs them.
    """
    class Autoencoder(nn.Module):
        def __init__(self):
            super().__init__()
            self.encoder = nn.Sequential(
                nn.Linear(input_dim, 128), nn.ReLU(),
                nn.Linear(128, 64), nn.ReLU(),
                nn.Linear(64, bottleneck_dim)
            )
            self.decoder = nn.Sequential(
                nn.Linear(bottleneck_dim, 64), nn.ReLU(),
                nn.Linear(64, 128), nn.ReLU(),
                nn.Linear(128, input_dim)
            )

        def forward(self, x):
            encoded = self.encoder(x)
            decoded = self.decoder(encoded)
            return encoded, decoded

    return Autoencoder()


def train_autoencoder(model, X_tensor, num_epochs=5000, learning_rate=0.0001, print_every=500):
    """Train the autoencoder and return the training losses."""
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    losses = []

    model.train()
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        encoded, decoded = model(X_tensor)
        loss = criterion(decoded, X_tensor)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if (epoch + 1) % print_every == 0:
            print(f"  Epoch {epoch+1}/{num_epochs}  \u2014  Loss: {loss.item():.4f}")

    print("  Training complete!")
    return losses


def get_embeddings(model, X_tensor):
    """Get the compressed 2D (or 3D) points from the trained autoencoder."""
    model.eval()
    with torch.no_grad():
        embeddings, _ = model(X_tensor)
    return embeddings.numpy()


def plot_clusters(embeddings, labels, label_names, title="Birdsong Clusters"):
    """Plot a 2D scatter plot colored by species."""
    plt.figure(figsize=(10, 7))
    for label in np.unique(labels):
        mask = labels == label
        plt.scatter(embeddings[mask, 0], embeddings[mask, 1],
                    label=label_names[label], alpha=0.7, s=60)
    plt.legend(fontsize=11)
    plt.title(title, fontsize=14)
    plt.xlabel("Dimension 1")
    plt.ylabel("Dimension 2")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


print("Helper functions ready.")

---
## Step 3: Load All Recordings

We load every WAV file and store the audio data alongside its species label.

In [ ]:
audio_data = []   # List of audio arrays
label_list = []   # Species label for each recording

label_encoder = LabelEncoder()
label_encoder.fit(SPECIES)
label_names = dict(zip(label_encoder.transform(SPECIES), SPECIES))

for species in SPECIES:
    folder = os.path.join(REPO_DIR, species)
    label = label_encoder.transform([species])[0]
    wav_files = sorted([f for f in os.listdir(folder) if f.endswith('.wav')])

    for fname in wav_files:
        try:
            y, _ = librosa.load(os.path.join(folder, fname), sr=SAMPLE_RATE, duration=10)
            audio_data.append(y)
            label_list.append(label)
        except Exception as e:
            print(f"  Error loading {species}/{fname}: {e}")

labels = np.array(label_list)
print(f"\nLoaded {len(audio_data)} recordings across {len(SPECIES)} species.")

---
## Section 1: What Are MFCCs?

To compare recordings, we need to turn each one into a fixed-size list of numbers. Raw audio won't work — recordings have different lengths, and the exact waveform values aren't very meaningful for comparison.

**MFCCs (Mel-Frequency Cepstral Coefficients)** solve this. Think of it this way:

> Two birds can sing the **same note**, but one sounds like a flute and one sounds like a kazoo. MFCCs measure **that difference** — the *texture* of the sound, not the pitch.

Technically, MFCCs capture the overall "shape" of the frequency spectrum at each moment. We extract 13 MFCCs from each recording, then summarize them as a mean and standard deviation, giving us **26 numbers per recording**.

Let's see what MFCCs look like for one recording.

In [ ]:
# Show MFCCs for one recording from each of two species
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

for i, sp_idx in enumerate([0, len(SPECIES)-1]):  # First and last species
    sp_name = SPECIES[sp_idx]
    # Find the first recording for this species
    rec_idx = next(j for j, lbl in enumerate(label_list) if lbl == sp_idx)
    mfcc = librosa.feature.mfcc(y=audio_data[rec_idx], sr=SAMPLE_RATE, n_mfcc=13)

    librosa.display.specshow(mfcc, sr=SAMPLE_RATE, x_axis='time', ax=axes[i])
    axes[i].set_title(f"MFCCs \u2014 {sp_name.replace('_', ' ').title()}", fontsize=12)
    axes[i].set_ylabel("MFCC #")

plt.tight_layout()
plt.show()

print("Each row is one MFCC coefficient over time.")
print("Different species produce different patterns \u2014 that's what the computer will learn from.")

### Extract MFCCs from Every Recording

In [ ]:
N_MFCC = 13
features = []

for audio in audio_data:
    mfcc = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC)
    # Summarize: mean and standard deviation of each coefficient
    feature_vec = np.concatenate([np.mean(mfcc, axis=1), np.std(mfcc, axis=1)])
    features.append(feature_vec)

X = np.array(features)

# Standardize to zero mean and unit variance (helps the neural network learn)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

print(f"Feature matrix: {X.shape[0]} recordings x {X.shape[1]} features")
print(f"Each recording is now represented by {X.shape[1]} numbers.")

---
## Section 2: The Autoencoder

We have 26 numbers per recording. That’s useful, but you can’t plot 26 dimensions on a screen. We need to squish them down to **just 2** so we can make a scatter plot.

That’s what an **autoencoder** does. Here’s the analogy:

> Imagine summarizing a **26-word description** of a birdsong into just an **X,Y point on a map**. The autoencoder learns to do this compression while preserving as much information as possible.

It works in two halves:
- **Encoder**: 26 features → 128 → 64 → **2** (the compressed point)
- **Decoder**: 2 → 64 → 128 → 26 (tries to reconstruct the original)

The network trains by trying to make the reconstruction match the original. The better it gets, the more meaningful those 2D points become.

In [ ]:
# Build and train the autoencoder (2D bottleneck)
print("Training autoencoder (bottleneck = 2 dimensions)...\n")

model_2d = build_autoencoder(input_dim=X_scaled.shape[1], bottleneck_dim=2)
losses_2d = train_autoencoder(model_2d, X_tensor, num_epochs=5000, learning_rate=0.0001)

In [ ]:
# Plot the training loss curve
plt.figure(figsize=(10, 3))
plt.plot(losses_2d, color='#3498db', linewidth=0.5)
plt.title("Training Loss Over Time")
plt.xlabel("Epoch")
plt.ylabel("Loss (lower = better reconstruction)")
plt.grid(True, alpha=0.3)
plt.show()

print("The loss goes down as the network learns to compress and reconstruct.")

In [ ]:
# Generate embeddings and plot
embeddings_2d = get_embeddings(model_2d, X_tensor)
plot_clusters(embeddings_2d, labels, label_names,
              title="2D Embedding of Birdsong MFCC Features (Autoencoder)")

print("Each dot is one recording, colored by species.")
print("Songs from the same species should cluster together!")

### Interpreting the Results

Look at the scatter plot and think about these questions:

- **Which species cluster tightly?** A tight cluster means those birds sing very consistently.
- **Which species overlap?** Overlap might mean two species sound similar — or that there’s a lot of variation within one species.
- **Why might two different species sound similar?** Related species (like Song sparrow and Swamp sparrow) share an evolutionary history.
- **What does a loose, spread-out cluster tell you?** That species might have a large song repertoire — lots of different song types.

---
## Section 3: Open Research

You've just built a working machine learning pipeline — from raw audio to a 2D map of birdsong. That's a real result. But it's also just a starting point.

In research, the next step is always: **can we do better?** That question opens up several directions, and each one is an active area of study in machine learning:

### Hyperparameter Tuning

The numbers we chose — learning rate (`0.0001`), number of epochs (`5000`), bottleneck size (`2`) — are called **hyperparameters**. We picked reasonable values, but we didn't prove they're the best. What happens if you change them?

- **Learning rate**: Try `0.001` or `0.00001`. Does the loss curve change? Do clusters get tighter or fall apart?
- **Epochs**: Try `500` vs `10000`. Is 5000 overkill, or not enough?
- **Bottleneck size**: Try `3` or `5` instead of `2`. Does the model reconstruct better with more room?

To test any of these, just change the numbers in the training cell above and re-run.

### Neural Architecture Search

Our autoencoder has a specific shape: 26 → 128 → 64 → 2 → 64 → 128 → 26. But why those layer sizes? Why three layers instead of two or five? This is called **architecture design**, and finding the best architecture for a given problem is a research field of its own.

Things you could try:
- Make the network wider (e.g., 256 → 128 → 2) or narrower (e.g., 32 → 16 → 2)
- Add or remove layers
- Try different activation functions (replace `nn.ReLU()` with `nn.Tanh()` or `nn.Sigmoid()`)

### New Designs

The autoencoder is just one approach. Researchers working on problems like this also explore:
- **Different features**: What if you used raw spectrograms instead of MFCCs? Or combined both?
- **Different models**: PCA, t-SNE, and UMAP can also reduce dimensions — how do they compare?
- **Supervised learning**: We never told the model which species is which. What if we did? That's classification, and it's a whole different approach.

### What "Research" Really Means

Research isn't about getting the right answer on the first try. It's about **systematically changing one thing at a time**, observing what happens, and building understanding from the pattern of results. That's exactly what you can do here — change a number, re-run, and look at the plot. Every run teaches you something about how the model works and what matters for separating birdsong.

---
## What You Learned

| Concept | Plain English |
|---------|---------------|
| **MFCCs** | Numbers that capture the *texture* of a sound, not just its pitch |
| **Autoencoder** | A neural network that learns to compress data into fewer dimensions while keeping the important parts |
| **Bottleneck** | The narrow middle layer \u2014 the smaller it is, the harder the compression |
| **Learning rate** | How big of a step the network takes when adjusting \u2014 too big or too small both fail |
| **Epochs** | Number of training passes \u2014 too few = underfitting, enough = convergence |
| **Decoder** | Forces the encoder to preserve useful information by requiring reconstruction |
| **Clustering** | Similar songs end up near each other in 2D space \u2014 species form groups |

**Next step:** Open **Notebook 4 \u2014 Web Audio** to download and explore birdsong from online sources!